In [22]:
# import modules
import os
import argparse
import logging
import numpy as np
import pandas as pd
from scipy.special import gammaln
from scipy import stats
import drugsets_func as df_func


In [23]:
# scz gsa out files
GSA_MAGZ="/Users/nyb/phd_code/drugsets-may2026/output/drugsets-out-magma-z-no-mhc/dgidb2-median/mp0/cond/scz.no.mhc.pgc3.magZ.cond.drug_mp0.gsa.out"
GSA_POPS="/Users/nyb/phd_code/drugsets-may2026/output/drugsets-out-pops-coexp-no-mhc/dgidb2-median/mp0/cond/scz.no.mhc.pgc3.coexp.cond.drug_mp0.gsa.out"

# scz null distributions
SCZ_MAGZ_NULL="/Users/nyb/phd_code/drugsets-may2026/data/null_set_distributions/scz.no.mhc.pgc3.drug.genome.magmaz.cond.null.dist.parquet"
SCZ_POPS_NULL="/Users/nyb/phd_code/drugsets-may2026/data/null_set_distributions/scz.no.mhc.pgc3.drug.genome.coexp.cond.null.dist.parquet"

In [24]:

GSA_REQUIRED = ["VARIABLE", "NGENES", "BETA", "SE", "P"]
DUP_COLS = ["NGENES", "BETA", "BETA_STD", "SE", "P"]


# ---------------------------------------------------------------------------
# reading inputs
# ---------------------------------------------------------------------------

def read_gsa_out(path):
    """Read a MAGMA .gsa.out file (header comment lines start with '#')."""
    df = pd.read_csv(path, sep=r"\s+", comment="#")
    missing = [c for c in GSA_REQUIRED if c not in df.columns]
    if missing:
        raise ValueError(f"[!] ERROR: {path} is missing columns: {', '.join(missing)}")
    # MAGMA truncates VARIABLE; FULL_NAME holds the untruncated name when present
    if "FULL_NAME" in df.columns:
        df["FULL_NAME"] = df["FULL_NAME"].fillna(df["VARIABLE"])
    else:
        df["FULL_NAME"] = df["VARIABLE"]
    df["FULL_NAME"] = df["FULL_NAME"].astype(str)
    return df


def read_exclude(path):
    """Sets to drop: a CSV/TSV with a 'full_name' column, or one name per line."""
    try:
        tab = pd.read_csv(path, sep=None, engine="python")
        if "full_name" in tab.columns:
            return set(tab["full_name"].astype(str))
    except Exception:
        pass
    with open(path) as fh:
        return {line.strip() for line in fh if line.strip()}


def load_null(path):
    """Load a null distribution of STAT values (.parquet, or whitespace-delimited text)."""
    if path.endswith(".parquet"):
        stat = pd.read_parquet(path, columns=["STAT"])["STAT"]  # needs pyarrow
    else:
        tab = pd.read_csv(path, sep=r"\s+", comment="#")
        if "STAT" in tab.columns:
            stat = tab["STAT"]
        else:  # headerless single column
            stat = pd.read_csv(path, sep=r"\s+", comment="#", header=None).iloc[:, 0]
    null = pd.to_numeric(stat, errors="coerce").to_numpy(float)
    n_na = int(np.isnan(null).sum())
    null = null[~np.isnan(null)]
    if null.size == 0:
        raise ValueError(f"[!] ERROR: null distribution is empty or all NA: {path}")
    logging.info(f"[+] INFO: Loaded null distribution: N = {null.size} ({n_na} NA dropped)")
    return null


# ---------------------------------------------------------------------------
# core steps
# ---------------------------------------------------------------------------

def collapse_duplicates(df, cols=DUP_COLS, keep="alpha"):
    """Collapse rows with identical values in `cols` into one row per unique test.

    The kept row is the alphabetically first FULL_NAME (keep="alpha", case-insensitive)
    or the shortest FULL_NAME (keep="shortest", ties broken alphabetically); its
    FULL_NAME, VARIABLE and all other columns are kept as they are.
    Adds:
      N_DUP       : number of sets sharing these results (1 = not duplicated)
      DUP_MEMBERS : all member FULL_NAMEs, alphabetically, ';'-separated
    """
    missing = [c for c in cols if c not in df.columns]
    if missing:
        raise ValueError(f"[!] ERROR: cannot check duplicates, missing columns: {', '.join(missing)}")
    if keep not in ("alpha", "shortest"):
        raise ValueError(f"[!] ERROR: keep must be 'alpha' or 'shortest', not {keep!r}")

    df = df.copy()
    # exact match on the values as read from file (NaN compares equal to NaN here)
    key = df[cols].astype(str).agg("|".join, axis=1)
    grp = df.groupby(key, sort=False)["FULL_NAME"]
    df["N_DUP"] = grp.transform("size")
    df["DUP_MEMBERS"] = grp.transform(lambda s: ";".join(sorted(s, key=lambda n: (n.lower(), n))))

    # order rows so the one to keep comes first within its group
    name_lower = df["FULL_NAME"].str.lower()
    sort_by = [name_lower, df["FULL_NAME"]]
    if keep == "shortest":
        sort_by.insert(0, df["FULL_NAME"].str.len())
    order = np.lexsort([s.to_numpy() for s in reversed(sort_by)])

    out = (df.iloc[order]
             .assign(_key=key.iloc[order].to_numpy())
             .drop_duplicates("_key", keep="first")
             .drop(columns="_key")
             .sort_index())  # back to original file order

    n_groups = int((out["N_DUP"] > 1).sum())
    logging.info(f"[+] INFO: {len(df)} sets collapsed to {len(out)} unique tests "
                 f"({n_groups} duplicate groups, {len(df) - len(out)} rows merged).")
    return out.reset_index(drop=True)


def empirical_p(stat, null):
    """One-sided (upper tail) empirical p = (M + 1) / (N + 1), where M is the number of
    null statistics >= the observed statistic. Matches compute_empirical_p() in the R code."""
    srt = np.sort(np.asarray(null, float))
    stat = np.asarray(stat, float)
    N = srt.size
    M = N - np.searchsorted(srt, stat, side="left")
    p = (M + 1) / (N + 1)
    return np.where(np.isnan(stat), np.nan, p)


# ---------------------------------------------------------------------------
# main entry point
# ---------------------------------------------------------------------------

def infer_set_type(gsa_file):
    """'drug' or 'moa' from the DRUGSETS label in the file name (e.g. .drug_mp0, .moa_group)."""
    name = os.path.basename(gsa_file)
    if ".drug_" in name:
        return "drug"
    if ".moa_" in name:
        return "moa"
    raise ValueError(f"[!] ERROR: cannot tell whether {name} holds drug or MoA sets; "
                     "pass set_type='drug' or set_type='moa'.")


def process_gsa(gsa_file, null_file=None, use_pops=False, min_size=2, exclude=None,
                hide=("druggable",), keep="alpha", method="fdr", alpha=0.05,
                set_type=None, out_file=None):
    """Clean, collapse duplicates, (empirically) re-test and correct one MAGMA .gsa.out file.

    gsa_file  : MAGMA .gsa.out
    null_file : null distribution of STAT; if given, P_EMP becomes the primary p
    use_pops  : True for PoPS runs; warns if no null is supplied
    min_size  : minimum NGENES to keep a set
    exclude   : iterable of FULL_NAMEs to drop before testing
    hide      : covariate names to drop (conditioned-on sets, gene covariates)
    keep      : which duplicate's name to keep: 'alpha' (first alphabetically) or 'shortest'
    method    : multiple-testing correction, 'fdr' (Benjamini-Hochberg, default) or 'bonf'
    alpha     : significance threshold on P_ADJ
    set_type  : 'drug' or 'moa'; inferred from the file name (.drug_* / .moa_*) if not given
    out_file  : output path (default: <prefix>.drugs.out or <prefix>.moa.out)
    """
    if method not in ("fdr", "bonf"):
        raise ValueError(f"[!] ERROR: method must be 'fdr' or 'bonf', not {method!r}")
    if set_type is None:
        set_type = infer_set_type(gsa_file)
    if set_type not in ("drug", "moa"):
        raise ValueError(f"[!] ERROR: set_type must be 'drug' or 'moa', not {set_type!r}")

    logging.info(f"[+] INFO: Post-processing {gsa_file}")
    df = read_gsa_out(gsa_file)
    n_read = len(df)

    # 1. drop covariates / conditioned-on sets
    if "TYPE" in df.columns:
        df = df[df["TYPE"].astype(str).str.upper() != "COVAR"]
    df = df[~df["FULL_NAME"].isin(set(hide or ()))]

    # 2. user exclusions and minimum set size (on NGENES actually analysed)
    if exclude:
        df = df[~df["FULL_NAME"].isin(set(exclude))]
    n_before_size = len(df)
    df = df[df["NGENES"] >= min_size].reset_index(drop=True)
    logging.info(f"[+] INFO: {n_read} rows read; {n_before_size - len(df)} sets dropped with "
                 f"NGENES < {min_size}; {len(df)} sets kept.")
    if df.empty:
        raise ValueError("[!] ERROR: no gene sets left after filtering.")

    # 3. collapse duplicated sets into one row per unique test
    n_total = len(df)
    df = collapse_duplicates(df, keep=keep)

    # 4. test statistic and p-values
    df["STAT"] = df["BETA"] / df["SE"]
    df = df.rename(columns={"P": "P_MAGMA"})
    if null_file:
        df["P_EMP"] = empirical_p(df["STAT"], load_null(null_file))
        df["P"] = df["P_EMP"]
        df["P_SOURCE"] = "empirical"
    else:
        if use_pops:
            logging.warning("[!] WARNING: PoPS run without a null distribution. Reporting MAGMA "
                            "p-values, which are anti-conservative for PoPS runs (P_SOURCE = magma).")
        df["P_EMP"] = np.nan
        df["P"] = df["P_MAGMA"]
        df["P_SOURCE"] = "magma"

    # 5. multiple testing over unique tests (NA p-values stay NA and are not counted)
    p = df["P"].to_numpy(float)
    ok = ~np.isnan(p)
    n_tests = int(ok.sum())
    p_adj = np.full(p.size, np.nan)
    if method == "bonf":
        p_adj[ok] = np.minimum(p[ok] * n_tests, 1.0)
    else:  # Benjamini-Hochberg
        p_ok = p[ok]
        order = np.argsort(p_ok)
        ranked = p_ok[order] * n_tests / np.arange(1, n_tests + 1)
        q = np.minimum.accumulate(ranked[::-1])[::-1]  # enforce monotonicity from the top
        q_ok = np.empty(n_tests)
        q_ok[order] = np.minimum(q, 1.0)
        p_adj[ok] = q_ok
    df["P_ADJ"] = p_adj
    df["N_TESTS"] = n_tests
    df["ADJ_METHOD"] = method
    df["SIG"] = df["P_ADJ"] < alpha  # NA -> False

    # 6. write
    cols = ["FULL_NAME", "VARIABLE", "TYPE", "NGENES", "BETA", "BETA_STD", "SE", "STAT",
            "P_MAGMA", "P_EMP", "P", "P_SOURCE", "P_ADJ", "ADJ_METHOD", "N_TESTS", "SIG",
            "N_DUP", "DUP_MEMBERS"]
    df = df[[c for c in cols if c in df.columns]].sort_values("P", kind="stable").reset_index(drop=True)

    if out_file is None:
        prefix = gsa_file
        for ext in (".gsa.out.txt", ".gsa.out"):
            if prefix.endswith(ext):
                prefix = prefix[: -len(ext)]
                break
        out_file = prefix + (".drugs.out" if set_type == "drug" else ".moa.out")

    n_sig = int(df["SIG"].sum())                         # significant unique tests
    n_sig_all = int(df.loc[df["SIG"], "N_DUP"].sum())   # ... counting every duplicate member
    method_name = "Benjamini-Hochberg FDR" if method == "fdr" else "Bonferroni"
    header = [
        "# DRUGSETS v2.1 post-processed gene-set results",
        f"# INPUT_FILE = {os.path.basename(gsa_file)}",
        f"# SET_TYPE = {set_type}",
        f"# MIN_NGENES = {min_size}",
        f"# TOTAL_SETS = {n_total}",
        f"# UNIQUE_SETS = {n_tests}",
        f"# P_SOURCE = {df['P_SOURCE'].iloc[0]}",
        f"# CORRECTION = {method} ({method_name})",
        f"# ALPHA = {alpha}",
        f"# SIGNIFICANT_UNIQUE_SETS = {n_sig}",
        f"# SIGNIFICANT_SETS_INCL_DUPLICATES = {n_sig_all}",
    ]
    with open(out_file, "w") as fh:
        fh.write("\n".join(header) + "\n")
        df.to_csv(fh, sep="\t", index=False, na_rep="NA", float_format="%.6g")

    logging.info(f"[+] INFO: {n_total} sets, {n_tests} unique; {n_sig} unique sets with "
                 f"{method} P_ADJ < {alpha}.")
    logging.info(f"[+] INFO: Saved {out_file}")
    return df


# ---------------------------------------------------------------------------
# standalone use
# ---------------------------------------------------------------------------

# def main():
#     ap = argparse.ArgumentParser(description="Post-process a DRUGSETS / MAGMA .gsa.out file.")
#     ap.add_argument("--gsa_file", required=True, help="MAGMA .gsa.out file")
#     ap.add_argument("--null_dist", help="Null distribution of STAT (.parquet or text)")
#     ap.add_argument("--use_pops", action="store_true", help="Results come from a PoPS run")
#     ap.add_argument("--min_size", type=int, default=2, help="Minimum NGENES (default 2)")
#     ap.add_argument("--exclude", help="File of set names to drop (one per line, or CSV with 'full_name')")
#     ap.add_argument("--hide", nargs="*", default=["druggable"], help="Covariate names to drop")
#     ap.add_argument("--keep", default="alpha", choices=["alpha", "shortest"],
#                     help="Which duplicate's name to keep (default: alphabetically first)")
#     ap.add_argument("--gsa_correct", default="fdr", choices=["fdr", "bonf"],
#                     help="Multiple-testing correction (default: fdr)")
#     ap.add_argument("--alpha", type=float, default=0.05)
#     ap.add_argument("--set_type", choices=["drug", "moa"],
#                     help="Drug or MoA sets (default: inferred from the file name)")
#     ap.add_argument("--out_file", help="Output path (default: <prefix>.drugs.out / <prefix>.moa.out)")
#     a = ap.parse_args()

#     process_gsa(a.gsa_file, null_file=a.null_dist, use_pops=a.use_pops, min_size=a.min_size,
#                 exclude=read_exclude(a.exclude) if a.exclude else None, hide=a.hide,
#                 keep=a.keep, method=a.gsa_correct, alpha=a.alpha, set_type=a.set_type,
#                 out_file=a.out_file)


# if __name__ == "__main__":
#     logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s",
#                         datefmt="%Y-%m-%d %H:%M:%S")
#     main()

In [25]:
# load gsa data
gsa_pops = process_gsa(GSA_POPS, null_file=SCZ_POPS_NULL, use_pops=True, min_size=5)

2026-10-18 13:44:02,715 - INFO - [+] INFO: Post-processing /Users/nyb/phd_code/drugsets-may2026/output/drugsets-out-pops-coexp-no-mhc/dgidb2-median/mp0/cond/scz.no.mhc.pgc3.coexp.cond.drug_mp0.gsa.out
2026-10-18 13:44:02,734 - INFO - [+] INFO: 7022 rows read; 4994 sets dropped with NGENES < 5; 2028 sets kept.
2026-10-18 13:44:02,799 - INFO - [+] INFO: 2028 sets collapsed to 1805 unique tests (79 duplicate groups, 223 rows merged).
2026-10-18 13:44:02,918 - INFO - [+] INFO: Loaded null distribution: N = 9733366 (0 NA dropped)
2026-10-18 13:44:03,364 - INFO - [+] INFO: 2028 sets, 1805 unique; 43 unique sets with fdr P_ADJ < 0.05.
2026-10-18 13:44:03,364 - INFO - [+] INFO: Saved /Users/nyb/phd_code/drugsets-may2026/output/drugsets-out-pops-coexp-no-mhc/dgidb2-median/mp0/cond/scz.no.mhc.pgc3.coexp.cond.drug_mp0.drugs.out


In [27]:
#!/usr/bin/env python3
"""
drugsets_null.py
Size-matched null gene sets for DRUGSETS empirical p-values.
Python port of generate_null_sets() in 02.6a_null_gene_sets.R.

Workflow for one null distribution:
  1. generate_null_sets()   -> MAGMA set file of random, size-matched gene sets
  2. run MAGMA on that set file with exactly the settings of the real run
     (same .genes.raw, same PoPS overwrite file, same conditioning / covariates)
  3. null_distribution()    -> STAT = BETA / SE of every null set, saved as .parquet,
                               which drugsets_post.process_gsa(null_file=...) reads

Genes are drawn from the genes MAGMA actually analysed (the GENE column of a
.genes.out file; for PoPS runs use the PoPS-adjusted .genes.out that DRUGSETS writes).
"""

import argparse                                  # command-line arguments for standalone use
import logging                                   # progress and warning messages
import os                                        # file paths and folders
import subprocess                                # running MAGMA
from concurrent.futures import ThreadPoolExecutor  # running MAGMA on several chunks at once

import numpy as np                               # arrays and random number generation
import pandas as pd                              # reading and writing tables
from scipy import stats                          # normal distribution, for the PoPS scores
from scipy.special import gammaln                # log-gamma, used to compute log(n choose k)


# ---------------------------------------------------------------------------
# inputs
# ---------------------------------------------------------------------------

def read_gene_pool(genes_out):
    """Unique, non-empty gene IDs from the GENE column of a MAGMA .genes.out file (file order)."""
    go = pd.read_csv(genes_out, sep=r"\s+", comment="#", usecols=["GENE"], dtype=str)  # read only the GENE column, as text
    genes = go["GENE"].dropna()                  # drop missing gene IDs
    genes = genes[genes != ""]                   # drop empty gene IDs
    return np.asarray(pd.unique(genes), dtype=object)  # unique genes, first-seen order, as a numpy array


def read_set_files(set_files):
    """(name, genes) for every line of one or more tab-delimited MAGMA set files."""
    if isinstance(set_files, (str, os.PathLike)):  # a single file was given
        set_files = [set_files]                  # wrap it in a list
    sets = []                                    # collects (set name, gene list) pairs
    for path in set_files:                       # each set file in turn
        with open(path) as fh:                   # open it for reading
            for line in fh:                      # one gene set per line
                parts = line.rstrip("\r\n").split("\t")  # strip the line ending and split on tabs
                if parts[0]:                     # skip lines with no set name (e.g. blank lines)
                    sets.append((parts[0], [g for g in parts[1:] if g]))  # name, then its non-empty genes
    return sets                                  # list of (name, genes) in file order


# ---------------------------------------------------------------------------
# sampling
# ---------------------------------------------------------------------------

def _draw_rows(rng, n_pool, s, k, max_cells=20_000_000):
    """k random s-subsets of range(n_pool) as a (k, s) array. Genes are unique within a row.

    Draws m > s indices per row with replacement and keeps the first s distinct ones in
    draw order, which is exactly sampling without replacement. Fast because s is small
    relative to n_pool (generate_null_sets drops sets above max_frac of the pool). The
    rare row with fewer than s distinct draws is redrawn with rng.choice(replace=False).
    """
    out = np.empty((k, s), dtype=np.int64)       # result: one row per null set, one column per gene
    target = min(s + 6 * np.sqrt(s) + 10, 0.9 * n_pool)  # distinct draws to aim for: s plus a wide safety margin
    m = max(s, int(np.ceil(np.log1p(-target / n_pool) / np.log1p(-1.0 / n_pool))))  # draws per row expected to give `target` distinct values
    rows_per_chunk = max(1, max_cells // m)      # rows per batch, to cap memory use

    filled = 0                                   # rows of `out` filled so far
    while filled < k:                            # until every set is drawn
        c = min(rows_per_chunk, k - filled)      # rows in this batch
        A = rng.integers(0, n_pool, size=(c, m))  # m random pool indices per row, with replacement
        order = np.argsort(A, axis=1, kind="stable")  # per row, positions that sort the values (ties keep draw order)
        sA = np.take_along_axis(A, order, axis=1)  # each row sorted
        first_sorted = np.ones(sA.shape, dtype=bool)  # mark first occurrences in the sorted rows...
        first_sorted[:, 1:] = sA[:, 1:] != sA[:, :-1]  # ...a value is new if it differs from the one before it
        first = np.empty_like(first_sorted)      # same marks, but in original draw order
        np.put_along_axis(first, order, first_sorted, axis=1)  # map the marks back to draw positions
        csum = np.cumsum(first, axis=1)          # running count of distinct values along each row
        ok = csum[:, -1] >= s                    # rows that got at least s distinct values
        keep = first & (csum <= s)               # the first s distinct values in each row

        block = out[filled:filled + c]           # the part of `out` this batch fills (a view, so writes go to `out`)
        block[ok] = A[ok][keep[ok]].reshape(-1, s)  # copy the kept values of the good rows
        for i in np.flatnonzero(~ok):            # rare rows with too few distinct values
            block[i] = rng.choice(n_pool, s, replace=False)  # draw them directly instead
        filled += c                              # move on to the next batch
    return out                                   # (k, s) array of pool indices


def _dup_rows(idx):
    """True for every row that repeats an earlier row as a set (gene order ignored)."""
    _, first = np.unique(np.sort(idx, axis=1), axis=0, return_index=True)  # sort genes in each row, find first copy of each distinct row
    dup = np.ones(len(idx), dtype=bool)          # start by marking every row as a duplicate
    dup[first] = False                           # unmark the first copy of each distinct set
    return dup                                   # True = repeats an earlier set


# ---------------------------------------------------------------------------
# main function
# ---------------------------------------------------------------------------

def _make_unique(rng, idx, n_pool, s, max_tries):
    """Redraw rows of idx (null sets of size s) that repeat an earlier row.
    Returns the updated idx and the number of duplicates left."""
    k = len(idx)                                                            # number of sets of this size
    dup = _dup_rows(idx)                                                    # True for each set that repeats an earlier one
    log_choose = gammaln(n_pool + 1) - gammaln(s + 1) - gammaln(n_pool - s + 1)  # log of the number of distinct sets of size s
    if log_choose < np.log(k):                                              # fewer distinct sets exist than were requested
        logging.warning(f"[!] WARNING: uniqueness impossible for size {s} "
                        f"({k} sets requested); {int(dup.sum())} duplicates kept.")  # warn and keep the duplicates
        return idx, int(dup.sum())                                          # nothing can be redrawn
    n_first, tries = int(dup.sum()), 0                                      # duplicates before redrawing; rounds used
    while dup.any() and tries < max_tries:                                  # redraw until unique or out of rounds
        idx[dup] = _draw_rows(rng, n_pool, s, int(dup.sum()))               # new random sets for duplicated rows only
        dup, tries = _dup_rows(idx), tries + 1                              # re-check and count the round
    if n_first:                                                             # report only if there was something to fix
        logging.info(f"  size {s}: {k:,} sets, {n_first:,} duplicate(s) fixed in {tries} round(s)"
                     + (f", {int(dup.sum())} STILL REMAIN" if dup.any() else ""))  # include any that remain
    return idx, int(dup.sum())                                              # updated sets and duplicates left


def generate_null_sets(set_files, genes_out, out_file, n_sets, cond_name="druggable",
                       write_cond=True, min_size=2, max_frac=0.1, unique_sets=False,
                       max_tries=50, seed=20260823):
    """Write a MAGMA set file of size-matched null gene sets, plus a key file.

    set_files   : one or more MAGMA set files. Used ONLY for the size distribution
                  (number of unique genes per set); no gene from these files is sampled.
    genes_out   : MAGMA .genes.out; its GENE column is the sampling pool.
    out_file    : output set file, e.g. 'pheno.drug.null.sets.txt'. The key is written
                  next to it as '<out_file minus .txt>.key.tsv'.
    n_sets      : number of null sets to make.
    cond_name   : name of the background (conditioning) row in set_files; not a real set.
    write_cond  : if True, also write a '<cond_name>' line (its genes restricted to the
                  pool) so MAGMA can condition on it exactly as in the real run.
    min_size    : real sets smaller than this are left out of the size table.
    max_frac    : real sets larger than this fraction of the gene pool are left out of the
                  size table, with a warning (default 0.1, i.e. 10% of the pool).
    unique_sets : if True, redraw null sets that duplicate another set of the same size.
    max_tries   : maximum redraw rounds per size (unique_sets only).
    seed        : RNG seed (numpy's generator, so draws differ from R's for the same seed).

    Returns the key as a DataFrame (null_set, source_set, size).
    """
    rng = np.random.default_rng(seed)                                       # seeded random number generator
    os.makedirs(os.path.dirname(os.path.abspath(out_file)), exist_ok=True)  # make the output folder if missing

    # ---- sampling pool ----
    pool = read_gene_pool(genes_out)                                        # unique genes in the .genes.out GENE column
    n_pool = len(pool)                                                      # number of genes to sample from
    if n_pool < 2:                                                          # a set needs at least 2 genes
        raise ValueError("[!] ERROR: gene pool has fewer than 2 usable genes.")  # stop: pool too small
    logging.info(f"[+] INFO: Sampling pool: {n_pool} genes from {os.path.basename(genes_out)}")  # report pool size

    # ---- sizes of the real sets ----
    if isinstance(set_files, (str, os.PathLike)):                           # a single file was given
        set_files = [set_files]                                             # wrap it in a list
    sets = read_set_files(set_files)                                        # (name, genes) for every line of every file
    real = pd.DataFrame([(n, len(set(g))) for n, g in sets if n != cond_name],
                        columns=["set", "size"])                            # name and unique gene count of each real set
    max_size = int(max_frac * n_pool)                                       # largest set size allowed
    small, big = real["size"] < min_size, real["size"] > max_size           # sets outside the allowed size range
    if big.any():                                                           # some real sets are too large
        logging.warning(f"[!] WARNING: {int(big.sum())} real set(s) larger than {max_size} genes "
                        f"({100 * max_frac:g}% of the pool) left out of the size table: "
                        f"{', '.join(real.loc[big, 'set'].head(5))}{' ...' if big.sum() > 5 else ''}")  # name up to 5 of them
    real = real[~small & ~big]                                              # keep sets within the allowed range
    if real.empty:                                                          # nothing left to take sizes from
        raise ValueError(f"[!] ERROR: no real sets between {min_size} and {max_size} genes.")  # stop
    names, sizes = real["set"].to_numpy(object), real["size"].to_numpy(np.int64)  # arrays for fast indexing
    logging.info(f"[+] INFO: Real sets: {len(sizes)} ({int(small.sum())} below {min_size} genes dropped) "
                 f"| size range: {sizes.min()}-{sizes.max()}")              # report the size table

    # ---- size table: every real set once, then resampled up to n_sets ----
    n_real = len(sizes)                                                     # number of real sets
    pick = (rng.choice(n_real, n_sets, replace=False) if n_sets <= n_real   # fewer nulls than real sets: no replacement
            else np.r_[np.arange(n_real), rng.choice(n_real, n_sets - n_real)])  # else each real set once, rest at random
    src_names, src_sizes = names[pick], sizes[pick]                         # source set and size of each null set

    # ---- background (conditioning) line ----
    cond_genes = {g for n, gs in sets if n == cond_name for g in gs}        # genes in the background row(s)
    cond = [g for g in pool if g in cond_genes] if write_cond else []       # those in the pool, in pool order
    if write_cond and not cond:                                             # line requested but no genes to write
        logging.warning(f"[!] WARNING: no '{cond_name}' genes found in the pool; "
                        "no background line written.")                      # warn

    # ---- draw and write, one size at a time (different sizes can never be duplicates) ----
    logging.info(f"[+] INFO: Drawing {n_sets:,} null sets...")              # progress message
    n_done, n_dup_left = 0, 0                                               # sets written so far; duplicates kept
    with open(out_file, "w") as fh:                                         # open the output set file
        for s in np.unique(src_sizes):                                      # each distinct size, smallest first
            k = int((src_sizes == s).sum())                                 # number of null sets of this size
            idx = _draw_rows(rng, n_pool, int(s), k)                        # k random sets as pool indices, no gene twice
            if unique_sets and k > 1:                                       # optionally remove duplicated sets
                idx, left = _make_unique(rng, idx, n_pool, int(s), max_tries)  # redraw duplicates
                n_dup_left += left                                          # tally any that could not be fixed
            if s > 1 and np.any(np.diff(np.sort(idx, axis=1), axis=1) == 0):  # any gene twice within a set?
                raise RuntimeError("[!] Internal error: a null set contains a repeated gene.")  # should never happen
            fh.writelines(f"null.s{n_done + i + 1}\t" + "\t".join(row) + "\n"
                          for i, row in enumerate(pool[idx]))               # one line per set: name, then its genes
            n_done += k                                                     # advance the set counter
        if cond:                                                            # background genes to write
            fh.write("\t".join([cond_name] + cond) + "\n")                  # background line at the end

    if unique_sets:                                                         # summarise duplicate handling
        logging.info(f"[+] INFO: Duplicate sets remaining: {n_dup_left} "
                     f"({100 * n_dup_left / n_sets:.3g}%)")                 # count and percentage
    logging.info("[+] INFO: Check passed: no repeated genes within any null set.")  # every size group passed
    if cond:                                                                # background line was written
        logging.info(f"[+] INFO: Background line '{cond_name}' written ({len(cond)} genes).")  # report its size

    # ---- key: which real set each null set took its size from ----
    order = np.argsort(src_sizes, kind="stable")                            # file order: by size, then table order
    key = pd.DataFrame({"null_set": [f"null.s{i + 1}" for i in range(n_sets)],  # null set names, as in the file
                        "source_set": src_names[order],                     # real set whose size was used
                        "size": src_sizes[order]})                          # size of the null set
    key_file = (out_file[:-4] if out_file.endswith(".txt") else out_file) + ".key.tsv"  # key path next to the set file
    key.to_csv(key_file, sep="\t", index=False)                             # save the key
    logging.info(f"[+] INFO: Saved {out_file} and {key_file}")              # report the outputs
    return key                                                              # key as a DataFrame


# ---------------------------------------------------------------------------
# after MAGMA has been run on the null sets
# ---------------------------------------------------------------------------

def null_distribution(null_gsa_out, out_file, cond_name="druggable"):
    """STAT = BETA / SE for every null set in one or more MAGMA .gsa.out files; saved for process_gsa().

    Written as .parquet (needs pyarrow) when out_file ends in .parquet, otherwise as a
    tab-delimited text file (.tsv / .tsv.gz). Both can be read by drugsets_post.load_null().
    """
    if isinstance(null_gsa_out, (str, os.PathLike)):           # a single .gsa.out was given
        null_gsa_out = [null_gsa_out]                          # wrap it in a list
    parts = []                                                 # one table per .gsa.out
    for path in null_gsa_out:                                  # each MAGMA results file (one per chunk)
        df = pd.read_csv(path, sep=r"\s+", comment="#")        # MAGMA results for the null sets, '#' header lines skipped
        name = df["FULL_NAME"] if "FULL_NAME" in df.columns else df["VARIABLE"]  # untruncated set names if MAGMA wrote them
        keep = name.astype(str) != cond_name                   # drop the background (conditioning) row
        if "TYPE" in df.columns:                               # results include a TYPE column
            keep &= df["TYPE"].astype(str).str.upper() != "COVAR"  # also drop gene covariates
        df = df[keep]                                          # null sets only
        parts.append(pd.DataFrame({"SET": name[keep].astype(str).to_numpy(),  # null set name
                                   "NGENES": df["NGENES"].to_numpy(),  # genes MAGMA used for that set
                                   "STAT": (df["BETA"] / df["SE"]).to_numpy()}))  # test statistic, same as for the real sets
    null = pd.concat(parts, ignore_index=True)                 # all chunks in one table
    if out_file.endswith(".parquet"):                          # parquet requested
        null.to_parquet(out_file, index=False)                 # save as parquet (needs pyarrow)
    else:                                                      # any other extension
        null.to_csv(out_file, sep="\t", index=False)           # save as tab-delimited text (gzipped if it ends in .gz)
    logging.info(f"[+] INFO: Null distribution: {len(null):,} statistics "
                 f"({int(null['STAT'].isna().sum())} NA) saved to {out_file}")  # report size, NAs and path
    return null                                                # the null distribution as a DataFrame


# ---------------------------------------------------------------------------
# full pipeline for PoPS runs: null sets -> MAGMA -> parquet
# ---------------------------------------------------------------------------

def make_pops_genes_out(magma_genes_out, pops_preds, out_file, adj_pops="yes"):
    """PoPS-substituted .genes.out, made exactly as DrugSetAnalysis.normalize_pops_and_adjust_genes_out()."""
    pops_df = pd.read_csv(pops_preds, sep="\t")                # PoPS predictions (ENSGID, PoPS_Score, ...)
    if adj_pops == "yes":                                      # rank-based inverse normal transform
        ranks = pops_df["PoPS_Score"].rank(method="average")   # ranks, ties averaged
        n, c = len(pops_df), 3 / 8                             # number of genes; Blom's constant
        pops_df["normed_pops"] = stats.norm.ppf((ranks - c) / (n - 2 * c + 1))  # Blom's formula -> normal scores
    else:                                                      # plain z-scores
        pops_df["normed_pops"] = stats.zscore(pops_df["PoPS_Score"])  # (score - mean) / sd
    pops_df["pvalue"] = 2 * stats.norm.sf(abs(pops_df["normed_pops"]))  # two-sided p-value of the normal score
    magma_df = pd.read_csv(magma_genes_out, delimiter=r"\s+", comment="#")  # original MAGMA gene results
    merged = magma_df.merge(pops_df[["ENSGID", "normed_pops", "pvalue"]],
                            left_on="GENE", right_on="ENSGID", how="inner")  # keep genes present in both
    merged = merged.drop(columns=["ENSGID", "ZSTAT", "P"]).rename(
        columns={"normed_pops": "ZSTAT", "pvalue": "P"})       # PoPS scores replace MAGMA's ZSTAT and P
    merged.to_csv(out_file, sep="\t", index=False)             # save, used as MAGMA's overwrite= file
    logging.info(f"[+] INFO: PoPS genes.out: {len(merged)} genes saved to {out_file}")  # report
    return out_file                                            # path to the new file


def _split_set_file(set_file, chunk_size, cond_name):
    """Split a null set file into files of chunk_size sets; the background line goes into every chunk."""
    stem = set_file[:-4] if set_file.endswith(".txt") else set_file  # base name for the chunk files
    paths, fh, n, cond_line = [], None, 0, None                # chunk paths; open chunk; sets written; background line
    with open(set_file) as src:                                # read the full null set file
        for line in src:                                       # one set per line
            if line.startswith(cond_name + "\t"):              # the background line
                cond_line = line                               # keep it for every chunk
                continue                                       # not a null set
            if n % chunk_size == 0:                            # time to start a new chunk
                if fh:                                         # a chunk is open
                    fh.close()                                 # close it
                paths.append(f"{stem}.chunk{len(paths) + 1:03d}.txt")  # name of the next chunk
                fh = open(paths[-1], "w")                      # open it
            fh.write(line)                                     # copy the set into the current chunk
            n += 1                                             # count it
    if fh:                                                     # last chunk still open
        fh.close()                                             # close it
    if cond_line:                                              # sets are conditioned on the background
        for path in paths:                                     # every chunk...
            with open(path, "a") as fh:                        # ...opened for appending
                fh.write(cond_line)                            # gets the background line at the end
    logging.info(f"[+] INFO: Split {n:,} null sets into {len(paths)} chunks of up to {chunk_size:,}")  # report
    return paths                                               # chunk file paths


def _null_magma_cmd(magma, raw_file, pops_genes_out, set_file, out, conditional, cond_name, magma_cov):
    """MAGMA gene-set command for null sets, mirroring DrugSetAnalysis.build_magma_cmd() for PoPS runs."""
    cmd = [magma, "--gene-results", raw_file, f"overwrite={pops_genes_out}",
           "--set-annot", set_file]                            # PoPS-substituted gene results + null sets
    hide = []                                                  # everything to condition on and hide
    if magma_cov:                                              # gene-level covariates used in the real run
        cmd += ["--gene-covar", magma_cov[0], f"filter-read={','.join(magma_cov[1:])}"]  # same covariate file and columns
        hide += list(magma_cov[1:])                            # condition on them too
    if conditional == "yes":                                   # real run conditioned on druggable genes
        hide.append(cond_name)                                 # so do the null sets
    if hide:                                                   # anything to condition on
        cmd += ["--model", f"condition-hide={','.join(hide)}"]  # one condition-hide, as in the real run
    return cmd + ["--out", out]                                # output prefix -> <out>.gsa.out


def _run_magma(cmd, out):
    """Run one MAGMA command; return its .gsa.out path, or raise with the end of the MAGMA log."""
    logging.info(f"[+] INFO: MAGMA started: {os.path.basename(out)}")  # progress message
    res = subprocess.run(cmd, capture_output=True, text=True)  # run MAGMA, keep its screen output
    if res.returncode != 0:                                    # MAGMA failed
        log = out + ".log"                                     # MAGMA's own log file
        tail = open(log).read()[-2000:] if os.path.isfile(log) else (res.stdout + res.stderr)[-2000:]  # last part of the log
        raise RuntimeError(f"[!] ERROR: MAGMA failed for {out} (exit {res.returncode}). "
                           f"Command: {' '.join(cmd)}\n{tail}")  # stop, showing the command and the error
    logging.info(f"[+] INFO: MAGMA finished: {os.path.basename(out)}")  # progress message
    return out + ".gsa.out"                                    # MAGMA's results file


def run_null_pops(raw_file, genes_out, pops_preds, set_files, out_prefix, n_sets=10_100_000,
                  magma="magma", adj_pops="yes", conditional="yes", magma_cov=None,
                  cond_name="druggable", min_size=2, max_frac=0.1, seed=20260823,
                  chunk_size=1_000_000, n_jobs=1, keep_files=False):
    """Null distribution for a PoPS-based DRUGSETS run: null sets -> MAGMA -> <out_prefix>.null.dist.parquet.

    raw_file    : MAGMA .genes.raw used in the real run
    genes_out   : original MAGMA .genes.out (as given to --use_pops)
    pops_preds  : PoPS .preds file (as given to --use_pops)
    set_files   : real drug or MoA set file(s) of the real run; sizes only
    out_prefix  : path + prefix for all outputs, e.g. '.../scz.drug.genome.coexp.cond'
    n_sets      : number of null sets
    magma       : path to the MAGMA executable
    adj_pops    : 'yes' (inverse normal transformed PoPS scores) or 'no' (z-scores), as --adj_pops
    conditional : 'yes' to condition on the druggable set, as --conditional
    magma_cov   : [cov_file, covariate, ...] as --magma_cov, or None
    chunk_size  : null sets per MAGMA run (None = one run for everything)
    n_jobs      : MAGMA runs at the same time
    keep_files  : keep the null set files and null .gsa.out files (large) after the parquet is written

    Returns the path to the parquet file.
    """
    if conditional not in ("yes", "no"):                       # same choices as --conditional
        raise ValueError(f"[!] ERROR: conditional must be 'yes' or 'no', not {conditional!r}")  # stop
    if conditional == "yes" and not any(n == cond_name for n, _ in read_set_files(set_files)):  # nothing to condition on
        raise ValueError(f"[!] ERROR: conditional='yes' but the set file(s) have no '{cond_name}' row; "
                         "use the *_cond_sets.txt files or conditional='no'.")  # stop before any work is done
    os.makedirs(os.path.dirname(os.path.abspath(out_prefix)), exist_ok=True)  # make the output folder if missing

    # 1. PoPS-substituted gene results: the overwrite file for MAGMA, and the sampling pool
    pops_genes_out = make_pops_genes_out(genes_out, pops_preds, f"{out_prefix}.pops.genes.out", adj_pops)  # same as the real run

    # 2. size-matched null sets drawn from the genes in the PoPS run
    set_file = f"{out_prefix}.null.sets.txt"                   # all null sets in one file
    generate_null_sets(set_files, pops_genes_out, set_file, n_sets, cond_name=cond_name,
                       write_cond=(conditional == "yes"), min_size=min_size,
                       max_frac=max_frac, seed=seed)           # background line only for conditioned runs

    # 3. optionally split into chunks, so MAGMA runs are smaller and can run side by side
    if chunk_size and n_sets > chunk_size:                     # more sets than one chunk holds
        chunks = _split_set_file(set_file, chunk_size, cond_name)  # chunk files, each with the background line
    else:                                                      # everything in one run
        chunks = [set_file]                                    # the full file is the only chunk

    # 4. run MAGMA on every chunk with the real run's settings
    outs = [c[:-4] if c.endswith(".txt") else c for c in chunks]  # MAGMA --out prefix per chunk
    cmds = [_null_magma_cmd(magma, raw_file, pops_genes_out, c, o, conditional, cond_name, magma_cov)
            for c, o in zip(chunks, outs)]                     # one MAGMA command per chunk
    logging.info(f"[+] INFO: Running MAGMA on {len(cmds)} null set file(s), {n_jobs} at a time")  # progress message
    with ThreadPoolExecutor(max_workers=n_jobs) as pool:       # up to n_jobs MAGMA processes at once
        gsa_files = list(pool.map(_run_magma, cmds, outs))     # wait for all; any failure raises here

    # 5. collect the null statistics into one parquet file
    null_file = f"{out_prefix}.null.dist.parquet"              # output for process_gsa(null_file=...)
    null_distribution(gsa_files, null_file, cond_name=cond_name)  # STAT = BETA / SE of every null set

    # 6. remove the large intermediate files
    if not keep_files:                                         # unless asked to keep them
        for path in set(chunks + [set_file] + gsa_files):      # null set files and null MAGMA results
            if os.path.isfile(path):                           # still there
                os.remove(path)                                # delete it
        logging.info("[+] INFO: Removed null set files and null .gsa.out files (keep_files=False)")  # report
    return null_file                                           # path to the parquet


# ---------------------------------------------------------------------------
# standalone use
# ---------------------------------------------------------------------------

# def main():
#     ap = argparse.ArgumentParser(description="Generate size-matched null gene sets for DRUGSETS.")  # command-line parser
#     ap.add_argument("--set_file", nargs="+", required=True,
#                     help="Real MAGMA set file(s); used only for the size distribution")  # one or more real set files
#     ap.add_argument("--genes_out", required=True, help="MAGMA .genes.out defining the gene pool")  # sampling pool
#     ap.add_argument("--out", required=True, help="Output null set file (.txt)")  # where to write the null sets
#     ap.add_argument("--n_sets", type=int, required=True, help="Number of null sets")  # how many null sets
#     ap.add_argument("--cond_name", default="druggable", help="Background row name (default: druggable)")  # background set name
#     ap.add_argument("--no_cond", action="store_true", help="Do not write a background line")  # unconditioned runs
#     ap.add_argument("--min_size", type=int, default=2, help="Minimum set size (default 2)")  # smallest real set size used
#     ap.add_argument("--max_frac", type=float, default=0.1,
#                     help="Drop real sets larger than this fraction of the pool (default 0.1)")  # size limit
#     ap.add_argument("--unique_sets", action="store_true", help="Redraw duplicated null sets")  # remove duplicate null sets
#     ap.add_argument("--max_tries", type=int, default=50)       # redraw rounds per size
#     ap.add_argument("--seed", type=int, default=20260823)      # random seed
#     a = ap.parse_args()                                        # read the arguments

#     generate_null_sets(a.set_file, a.genes_out, a.out, a.n_sets, cond_name=a.cond_name,
#                        write_cond=not a.no_cond, min_size=a.min_size, max_frac=a.max_frac,
#                        unique_sets=a.unique_sets,
#                        max_tries=a.max_tries, seed=a.seed)     # make the null sets


# if __name__ == "__main__":                                     # only when run as a script, not when imported
#     logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s",
#                         datefmt="%Y-%m-%d %H:%M:%S")           # show INFO messages with a timestamp
#     main()                                                     # run

In [21]:
import os
print(os.getcwd())   

/Users/nyb/drugsets


In [19]:
# scz genes.raw and genes.out
SCZ_OUT="/Users/nyb/phd_code/drugsets-may2026/output/magma-v110-s1s2-pops-no-mhc/scz.no.mhc.pgc3.step2.genes.out"
SCZ_RAW="/Users/nyb/phd_code/drugsets-may2026/output/magma-v110-s1s2-pops-no-mhc/scz.no.mhc.pgc3.step2.genes.raw"

# scz pops .preds file
SCZ_POPS="/Users/nyb/phd_code/drugsets-may2026/pops-out-no-mhc/pops-coexp/scz.no.mhc.pgc3.coexp.preds"

# drug sets path
DRUG_SETS="/Users/nyb/drugsets/DATA/GENESETS/drug_mp0_dgidb_median_ensembl_cond_sets.txt"

# drug sets path
MOA_SETS="/Users/nyb/drugsets/DATA/GENESETS/moa_targ_chembl_group_ensembl_cond_sets.txt"

In [28]:
# generate_null_sets(set_files=DRUG_SETS, genes_out=SCZ_OUT, out_file="/Users/nyb/Downloads/druggable_null_sets.txt", n_sets=1000, cond_name="druggable",
#                        write_cond=True, min_size=2, max_frac=0.1, unique_sets=False,
#                        max_tries=50, seed=20260823)

In [30]:
# ---- settings ----
TEST_DIR = "/Users/nyb/Desktop/drugsets-testing-2026"      # all outputs go here
MAGMA = "magma"                                            # MAGMA executable (full path if not on your PATH)
N_SETS = 10000                                             # small test; the paper used 10.1 million
MIN_SIZE = 2                                               # minimum NGENES in process_gsa, as in your notebook
 
# ---- inputs (from testing-2026.ipynb) ----
SCZ_OUT = "/Users/nyb/phd_code/drugsets-may2026/output/magma-v110-s1s2-pops-no-mhc/scz.no.mhc.pgc3.step2.genes.out"   # MAGMA gene results
SCZ_RAW = "/Users/nyb/phd_code/drugsets-may2026/output/magma-v110-s1s2-pops-no-mhc/scz.no.mhc.pgc3.step2.genes.raw"   # MAGMA .raw file
SCZ_POPS = "/Users/nyb/phd_code/drugsets-may2026/pops-out-no-mhc/pops-coexp/scz.no.mhc.pgc3.coexp.preds"               # PoPS predictions
DRUG_SETS = "/Users/nyb/drugsets/DATA/GENESETS/drug_mp0_dgidb_median_ensembl_cond_sets.txt"                             # real drug sets (conditioned version)
GSA_POPS = ("/Users/nyb/phd_code/drugsets-may2026/output/drugsets-out-pops-coexp-no-mhc/dgidb2-median/mp0/cond/"
            "scz.no.mhc.pgc3.coexp.cond.drug_mp0.gsa.out")                                                             # real PoPS drug set results
SCZ_POPS_NULL = ("/Users/nyb/phd_code/drugsets-may2026/data/null_set_distributions/"
                 "scz.no.mhc.pgc3.drug.genome.coexp.cond.null.dist.parquet")                                           # paper's null, for comparison only
 
PREFIX = os.path.join(TEST_DIR, "scz.no.mhc.pgc3.drug.genome.coexp.cond")  # prefix for every output file
 
 
def gsa_header(path, key):
    """Value of a '# KEY = value' line at the top of a MAGMA .gsa.out file (None if absent)."""
    with open(path) as fh:                                 # open the file
        for line in fh:                                    # header lines come first
            if not line.startswith("#"):                   # end of the header
                return None                                # key not found
            if line.startswith(f"# {key} ="):              # the line we want
                return line.split("=", 1)[1].strip()       # its value
 
 
# ---- logging: to the screen and to a log file in TEST_DIR ----
os.makedirs(TEST_DIR, exist_ok=True)                       # make the test folder if missing
logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s", datefmt="%H:%M:%S",
                    handlers=[logging.StreamHandler(),
                              logging.FileHandler(PREFIX + ".test.log", mode="w")],
                    force=True)                            # force=True also works inside Jupyter
 
for path in (SCZ_OUT, SCZ_RAW, SCZ_POPS, DRUG_SETS, GSA_POPS):  # every input must exist
    if not os.path.isfile(path):                           # missing input
        raise FileNotFoundError(path)                      # stop before doing any work
 
# ---- 1. null sets -> MAGMA -> parquet ----
null_file = run_null_pops(
    raw_file=SCZ_RAW, genes_out=SCZ_OUT, pops_preds=SCZ_POPS, set_files=DRUG_SETS,
    out_prefix=PREFIX, n_sets=N_SETS, magma=MAGMA,
    adj_pops="yes", conditional="yes",                     # same as the real coexp.cond run
    chunk_size=None, n_jobs=1,                             # small test: one MAGMA run
    keep_files=True)                                       # keep null sets and null .gsa.out to inspect
 
# ---- 2. empirical p-values for the real results, with the new null ----
res_new = process_gsa(GSA_POPS, null_file=null_file, use_pops=True, min_size=MIN_SIZE,
                      out_file=PREFIX + ".drug_mp0.drugs.out")  # written to TEST_DIR, not next to the real results
 
# ---- 3. sanity checks ----
print("\n==== CHECKS ====")
null_gsa = PREFIX + ".null.sets.gsa.out"                   # null MAGMA results (kept by keep_files=True)
n_pops = sum(1 for _ in open(PREFIX + ".pops.genes.out")) - 1  # genes in the PoPS genes.out (minus header)
print(f"Genes in PoPS genes.out (sampling pool): {n_pops}")
print(f"TOTAL_GENES, real PoPS run:  {gsa_header(GSA_POPS, 'TOTAL_GENES')}")   # genes MAGMA used in the real run
print(f"TOTAL_GENES, null run:       {gsa_header(null_gsa, 'TOTAL_GENES')}")   # should match the real run
 
null_new = pd.read_parquet(null_file)["STAT"].dropna()     # new null statistics
summ = {"new null": null_new}                              # distributions to summarise
if os.path.isfile(SCZ_POPS_NULL):                          # paper null available
    summ["paper null"] = pd.read_parquet(SCZ_POPS_NULL, columns=["STAT"])["STAT"].dropna()  # paper null statistics
q = [0.01, 0.05, 0.5, 0.95, 0.99]                          # quantiles to compare
table = pd.DataFrame({k: [len(v), v.mean(), v.std(), *v.quantile(q)] for k, v in summ.items()},
                     index=["N", "mean", "sd"] + [f"q{int(x * 100)}" for x in q])  # one column per null
print("\nNull STAT distributions (should look alike):")
print(table.round(3).to_string())
 
if os.path.isfile(SCZ_POPS_NULL):                          # compare empirical p-values from both nulls
    res_old = process_gsa(GSA_POPS, null_file=SCZ_POPS_NULL, use_pops=True, min_size=MIN_SIZE,
                          out_file=PREFIX + ".drug_mp0.papernull.drugs.out")  # same real results, paper null
    both = res_new.merge(res_old, on="FULL_NAME", suffixes=("_new", "_paper"))  # match sets by name
    r = stats.pearsonr(-np.log10(both["P_EMP_new"]), -np.log10(both["P_EMP_paper"]))[0]  # agreement of -log10 p values
    print(f"\nCorrelation of -log10(P_EMP), new vs paper null: {r:.4f} ({len(both)} sets)")
    print(f"Smallest possible P_EMP with the test null: {1 / (len(null_new) + 1):.2g} "
          f"(paper null: {1 / (len(summ['paper null']) + 1):.2g})")   # resolution limit of a small null
    top = both.nsmallest(10, "P_EMP_paper")[["FULL_NAME", "P_MAGMA_new", "P_EMP_paper", "P_EMP_new"]]  # top sets in the paper
    print("\nTop 10 sets by paper-null P_EMP:")
    print(top.rename(columns={"P_MAGMA_new": "P_MAGMA"}).to_string(index=False))
 
print(f"\nAll outputs are in {TEST_DIR}")
 


13:46:25 - [+] INFO: PoPS genes.out: 17942 genes saved to /Users/nyb/Desktop/drugsets-testing-2026/scz.no.mhc.pgc3.drug.genome.coexp.cond.pops.genes.out
13:46:25 - [+] INFO: Sampling pool: 17942 genes from scz.no.mhc.pgc3.drug.genome.coexp.cond.pops.genes.out
13:46:25 - [+] INFO: Real sets: 8151 (0 below 2 genes dropped) | size range: 2-109
13:46:25 - [+] INFO: Drawing 10,000 null sets...
13:46:25 - [+] INFO: Check passed: no repeated genes within any null set.
13:46:25 - [+] INFO: Background line 'druggable' written (4290 genes).
13:46:25 - [+] INFO: Saved /Users/nyb/Desktop/drugsets-testing-2026/scz.no.mhc.pgc3.drug.genome.coexp.cond.null.sets.txt and /Users/nyb/Desktop/drugsets-testing-2026/scz.no.mhc.pgc3.drug.genome.coexp.cond.null.sets.key.tsv
13:46:25 - [+] INFO: Running MAGMA on 1 null set file(s), 1 at a time
13:46:25 - [+] INFO: MAGMA started: scz.no.mhc.pgc3.drug.genome.coexp.cond.null.sets
13:46:34 - [+] INFO: MAGMA finished: scz.no.mhc.pgc3.drug.genome.coexp.cond.null.sets


==== CHECKS ====
Genes in PoPS genes.out (sampling pool): 17942
TOTAL_GENES, real PoPS run:  17541
TOTAL_GENES, null run:       17942


13:46:34 - [+] INFO: Post-processing /Users/nyb/phd_code/drugsets-may2026/output/drugsets-out-pops-coexp-no-mhc/dgidb2-median/mp0/cond/scz.no.mhc.pgc3.coexp.cond.drug_mp0.gsa.out
13:46:34 - [+] INFO: 7022 rows read; 0 sets dropped with NGENES < 2; 7022 sets kept.



Null STAT distributions (should look alike):
       new null   paper null
N     10000.000  9733366.000
mean     -0.019       -0.001
sd        1.151        1.147
q1       -2.818       -2.734
q5       -1.907       -1.855
q50      -0.016       -0.017
q95       1.859        1.901
q99       2.714        2.832


13:46:35 - [+] INFO: 7022 sets collapsed to 5061 unique tests (660 duplicate groups, 1961 rows merged).
13:46:35 - [+] INFO: Loaded null distribution: N = 9733366 (0 NA dropped)
13:46:35 - [+] INFO: 7022 sets, 5061 unique; 15 unique sets with fdr P_ADJ < 0.05.
13:46:35 - [+] INFO: Saved /Users/nyb/Desktop/drugsets-testing-2026/scz.no.mhc.pgc3.drug.genome.coexp.cond.drug_mp0.papernull.drugs.out



Correlation of -log10(P_EMP), new vs paper null: 0.9975 (5061 sets)
Smallest possible P_EMP with the test null: 0.0001 (paper null: 1e-07)

Top 10 sets by paper-null P_EMP:
          FULL_NAME      P_MAGMA  P_EMP_paper  P_EMP_new
          CLEMIZOLE 3.984600e-11     0.000005     0.0001
        GSK1702934A 5.730900e-11     0.000006     0.0001
         ATAGABALIN 1.255600e-10     0.000007     0.0001
               BTP2 3.074800e-10     0.000010     0.0001
                NI2 4.678600e-10     0.000013     0.0001
         GABAPENTIN 4.082700e-09     0.000032     0.0001
         TERODILINE 4.338600e-09     0.000032     0.0001
GABAPENTINENACARBIL 2.257800e-08     0.000067     0.0001
         TOPIRAMATE 2.870500e-08     0.000073     0.0001
               ACAA 3.940000e-08     0.000080     0.0001

All outputs are in /Users/nyb/Desktop/drugsets-testing-2026
